# APK 4720 / APK 6725

# Assignment # 8

Please submit your assignment as a Jupyter  notebook (.ipynb file). Start a new Jupyter notebook and name it "YourName_Assignment_8"

Replace *YourName* with your first name and last name

>BEFORE STARTING THE ASSIGNMENT, MAKE SURE THAT **ULTRALYTICS** IS INSTALLED IN YOUR ENVIRONMENT. 

Visit `docs.ultralytics.com/quickstart/` for instructions on how to install this package in your system.

Note, if you are using macOS, you will need to downgrade `numpy` to use ultralytics. Use the following command 
```
    !pip install "numpy<2"
```


## Learning Objectives

By the end of this assignment, students will be able to:

- Apply pretrained CNN models for:
    - Object recognizion
	- Object detection
	- Segmentation
	- Pose estimation
- Interpret model outputs (boxes, masks, keypoints)
- Compare different types of visual outputs
- Use pose estimation results to classify different postures


To complete this assignment, unzip the file `Assignment8.zip` included in GitHub. After unzipping the file, you will have access to a folder (`Assignment8`) that contains the models and data employed in this assignment.  

## Part A (1 point) -- Object recognition 

In this section, we will use a model trained on ImageNet to identify objects on an image. 

1. Load the model `yolo26s-cls.pt` 
```Python
    from ultralytics import YOLO
    model_recognition = YOLO(r'Assignment8/yolo26s-cls.pt')
```
2. Load the image `lift.jpg` and processes it with the model. 

The model outputs the probability that the objects included in the ImageNet dataset are present in the image. 

Assuming that you results are stored in the variable `results`, you can list the top 5 objects identified, along with their probability using the following code:

```Python
    probs = results[0].probs
    names = results[0].names
    print("Top predictions:")
    for i in probs.top5:
        print(f"- {names[i]} ({float(probs.data[i]):.4f})")
```


### Questions:
1. What object (s) were detected by the model? 
2. Explain your results? Were the results expected/unexpected? 

## Part B (1 point) -- Object detection

In this section, we will use a model trained on COCO dataset to detect and localize objects in an image. 

1. Load the model `yolo26s.pt` 
```Python
    from ultralytics import YOLO
    model_object = YOLO(r'Assignment8/yolo26s.pt')
```
2. Load the image `lift.jpg` and processes it with the model. 

The model outputs the position of the different objects localized in the image. The model also produces the identity of the identified object and its probability. 

Plot the image along with the position and identity of the objects detected in the image. 

Hint: Review the notebook `07.01 CNN applications.ipynb` for examples on how to display the image and bounding boxes provided by the model. 

### Questions:
1. What object (s) were detected by the model? 
2. Explain your results? Were the results expected/unexpected? 

## Part C (1 point) -- Object segmentation

In this section, we will use a model trained on COCO segmentation dataset to detect, localize, and segment objects in an image. 

1. Load the model `yolo26s.pt` 
```Python
    from ultralytics import YOLO
    model_object = YOLO(r'Assignment8/yolo26s-seg.pt')
```
2. Load the image `lift.jpg` and processes it with the model. 


The model outputs a *mask* assigning a class to each pixel in the image. 

Assuming that your results are stored in the variable `results`, and your image is stored in the variable `img`, you can visualize your results using the following code:

```Python
    #convert image to numpy array
    img_np = np.array(img)

    #get the mask from the results
    mask = results[0].masks.data.cpu().numpy()  # (n_instances, h, w)
    #turn the stack of masks into a single binary mask
    mask_stack = results[0].masks.data.cpu().numpy()  # (n_instances, h, w)
    combined_mask = (mask_stack.max(axis=0) > 0.5).astype(np.uint8)

    #resize the mask to match the original image size if necessary
    combined_mask = cv2.resize(combined_mask, (img_np.shape[1], img_np.shape[0]), interpolation=cv2.INTER_NEAREST)

    #combine the original image with the mask to create an overlay
    
    color = np.array([255, 0, 0], dtype=np.uint8)
    alpha = 0.4
    overlay = img_np.copy()
    overlay[combined_mask == 1] = ((1 - alpha) * overlay[combined_mask == 1] + alpha * color).astype(np.uint8)

    #plot the original image, the mask, and the combined overlay
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img_np)
    axes[0].set_title("Original Image")
    axes[0].axis("off") 
    axes[1].imshow(combined_mask, cmap="gray")
    axes[1].set_title("Mask")
    axes[1].axis("off")
    axes[2].imshow(overlay)
    axes[2].set_title("Image + Mask")
    axes[2].axis("off")
    plt.tight_layout()
```

### Questions:
1. What object (s) were detected by the model? 
2. Explain your results? Were the results expected/unexpected? 

## Part D (1 point) -- Pose estimation

In this section, we will use a model trained on COCO Pose dataset to detect and localize the position of 17 body landmarks, including:

- COCO body landmarks:
    - nose
    - left_eye
    - right_eye
    - left_ear
    - right_ear
    - left_shoulder
    - right_shoulder
    - left_elbow
    - right_elbow
    - left_wrist
    - right_wrist
    - left_hip
    - right_hip
    - left_knee
    - right_knee
    - left_ankle
    - right_ankle

1. Load the model `yolo26s-pose.pt` 
```Python
    from ultralytics import YOLO
    model_pose = YOLO(r'Assignment8/yolo26s-pose.pt')
```
2. Load the image `lift.jpg` and processes it with the model. 

The model outputs the position of *persons* in the image, along with the *(x,y)* position of the different landmarks. The model also provides a *confidence* score, indicating if the body part is visible or not. 

Assuming that your results are stored in the variable `results`, and your image is stored in the variable `img`, you can visualize your results using the following code:

```Python
    plt.imshow(img)
    boxes = results[0].boxes.xyxy.cpu().numpy() #get the bounding boxes in xyxy format and convert to numpy array
    classes = results[0].boxes.cls.cpu().numpy() #get the class labels of the detected objects and convert to numpy array

    #plot the boxes and class labels on the image
    for idx, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        plt.gca().add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='red', linewidth=2)) #draw the bounding box on the image
        plt.text(x1, y1, f'{model.names[int(classes[idx])]}', color='red', fontsize=12) #add the class label to the bounding box

    #plot the landmarks 
    landmarks = results[0].keypoints.cpu().numpy() #get the keypoints (landmarks) of the detected objects and convert to numpy array
    for idx, landmark in enumerate(landmarks.data):
        for point in landmark:
            x, y, conf = point
            if conf > 0.5: #only plot the landmark points with confidence greater than 0.5
                plt.scatter(x, y, c='tab:orange', s=20, alpha = 0.8) #plot the landmark points on the image

    plt.show()
```

### Questions:
1. What object (s) were detected by the model? 
2. Explain your results? Were the results expected/unexpected? 

## Part E (1 point) -- Pose-based posture classification

In this section, you will use the output of the pose estimation model to classify whether a person is **sitting** or **standing**.

Instead of relying only on the model output, you will create a simple **rule-based classifier** based on the position of the body landmarks.

1. Use the model `yolo26s-pose.pt` to process the provided images of people sitting and standing.
2. Extract the coordinates of relevant landmarks, such as:
    - shoulders
    - hips
    - knees
    - ankles
3. Define at least one rule based on the landmark positions to classify each person as **sitting** or **standing**.

For example, you may use:
- the angle at the knee
- the relative vertical position of the hips and knees
- the relative alignment of the torso and legs

Plot each image with the detected landmarks and report the predicted posture.

Hint: Assuming that your results are stored in the variable `results`, the following code will provide the *(x,y)* position of the different landmarks localized by the model 

```Python
    COCO_KEYPOINT_NAMES = [
        'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
        'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
        'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
        'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
    ]

    landmarks = results[0].keypoints.cpu().numpy()

    all_persons = []
    for person_idx, landmark in enumerate(landmarks.data):
        person_data = {'person': person_idx + 1}
        for kp_idx, point in enumerate(landmark):
            x, y, conf = point
            name = COCO_KEYPOINT_NAMES[kp_idx]
            person_data[f'{name}_x'] = x if conf > 0.5 else None
            person_data[f'{name}_y'] = y if conf > 0.5 else None
            person_data[f'{name}_conf'] = conf
        all_persons.append(person_data)

    df_landmarks = pd.DataFrame(all_persons)
    print(df_landmarks.to_string())
```

### Questions:
1. What rule did you use to classify sitting vs. standing?
2. Did your rule work for all images? Why or why not?
3. What are some limitations of using a simple rule-based method for this task?